# Coinbase order-book feature construction

This notebook turns the reconstructed `SOL-USD` order book and matching trades into a uniform one-row-per-second research dataset. Every feature uses information received by the row's `decision_time`; future information appears only in columns beginning with `target_`.

Three hours is enough to validate the pipeline, but it is not enough to establish that a signal will remain predictive in other market conditions.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data/ticks")
PRODUCT_ID = "SOL-USD"
FREQUENCY = "1s"
TARGET_HORIZONS = (1, 5, 30)

# Match the newest reconstructed book file to the trade capture with the same timestamp.
book_files = sorted(DATA_DIR.glob(f"coinbase_{PRODUCT_ID}_book_metrics_*.csv"))
if not book_files:
    raise FileNotFoundError("Run Coinbase_Order_Book_Reconstruction.ipynb first.")

BOOK_PATH = book_files[-1]
capture_stamp = BOOK_PATH.stem.split("_book_metrics_", 1)[1]
TRADES_PATH = DATA_DIR / f"coinbase_{PRODUCT_ID}_trades_{capture_stamp}.csv"
if not TRADES_PATH.exists():
    raise FileNotFoundError(f"Matching trades file not found: {TRADES_PATH}")

print(f"Book:   {BOOK_PATH.resolve()}")
print(f"Trades: {TRADES_PATH.resolve()}")

Book:   /Users/hankrugg/Fordham/Fall26/Crypto/trading/data/ticks/coinbase_SOL-USD_book_metrics_20261009T133847Z.csv
Trades: /Users/hankrugg/Fordham/Fall26/Crypto/trading/data/ticks/coinbase_SOL-USD_trades_20261009T133847Z.csv


## Create a causal one-second timeline

A row labeled `10:38:48` contains the latest reconstructed book received during `(10:38:47, 10:38:48]`. Labeling on the right edge ensures the row is not treated as available before its underlying message arrived. Empty seconds carry the last known book forward.

In [2]:
book = pd.read_csv(BOOK_PATH, parse_dates=["event_time", "received_at"])
book = book.sort_values("received_at").drop_duplicates("received_at", keep="last")
book["source_received_at"] = book["received_at"]

# closed='right', label='right' prevents a message received after a boundary
# from being placed in a row labeled before it was known.
book_1s = (
    book.set_index("received_at")
    .resample(FREQUENCY, closed="right", label="right")
    .last()
    .ffill()
)
book_1s.index.name = "decision_time"

print(f"Uniform book rows: {len(book_1s):,}")
display(book_1s[["event_time", "source_received_at", "mid_price", "spread", "imbalance_10"]].head())

Uniform book rows: 10,800


,event_time,source_received_at,mid_price,spread,imbalance_10
decision_time,,,,,
2026-10-09 10:38:47+00:00,2026-10-09 10:38:46.321411+00:00,2026-10-09 10:38:46.505635+00:00,109.61,0.02,0.145982
2026-10-09 10:38:48+00:00,2026-10-09 10:38:47.536561+00:00,2026-10-09 10:38:47.554103+00:00,109.61,0.02,0.191700
2026-10-09 10:38:49+00:00,2026-10-09 10:38:48.477619+00:00,2026-10-09 10:38:48.583365+00:00,109.61,0.02,0.362584
2026-10-09 10:38:50+00:00,2026-10-09 10:38:49.531141+00:00,2026-10-09 10:38:49.544308+00:00,109.61,0.02,0.263517
2026-10-09 10:38:51+00:00,2026-10-09 10:38:50.496621+00:00,2026-10-09 10:38:50.545895+00:00,109.61,0.02,0.269830


## Aggregate trades by local receipt time

The initial `snapshot` of recent trades is excluded because those trades occurred before this capture began. Coinbase reports the maker side, so a `SELL` maker implies a buyer-initiated trade and a `BUY` maker implies a seller-initiated trade. Positive signed volume therefore represents buyer-initiated volume.

In [3]:
trades = pd.read_csv(TRADES_PATH, parse_dates=["exchange_time"])
trades = trades[trades["event_type"] == "update"].copy()
trades["received_at"] = pd.to_datetime(trades["received_at_ns"], unit="ns", utc=True)
trades["notional"] = trades["price"] * trades["size"]
trades["aggressor_sign"] = trades["maker_side"].map({"SELL": 1, "BUY": -1})
trades["signed_size"] = trades["size"] * trades["aggressor_sign"]

trade_1s = (
    trades.set_index("received_at")
    .resample(FREQUENCY, closed="right", label="right")
    .agg(
        trade_count=("trade_id", "nunique"),
        trade_volume=("size", "sum"),
        signed_trade_volume=("signed_size", "sum"),
        trade_notional=("notional", "sum"),
    )
)
trade_1s["trade_vwap"] = trade_1s["trade_notional"] / trade_1s["trade_volume"].replace(0, np.nan)

data = book_1s.join(trade_1s, how="left")
zero_when_no_trade = ["trade_count", "trade_volume", "signed_trade_volume", "trade_notional"]
data[zero_when_no_trade] = data[zero_when_no_trade].fillna(0)
data["trade_count"] = data["trade_count"].astype("int64")

print(f"Update trades included: {len(trades):,}")
display(data[["mid_price", "trade_count", "trade_volume", "signed_trade_volume"]].head())

Update trades included: 14,515


,mid_price,trade_count,trade_volume,signed_trade_volume
decision_time,,,,
2026-10-09 10:38:47+00:00,109.61,0,0.000000,0.000000
2026-10-09 10:38:48+00:00,109.61,0,0.000000,0.000000
2026-10-09 10:38:49+00:00,109.61,0,0.000000,0.000000
2026-10-09 10:38:50+00:00,109.61,1,2.659022,2.659022
2026-10-09 10:38:51+00:00,109.61,1,0.109469,0.109469


## Construct features

All calculations below use the current row or earlier rows. `microprice_edge_bps` shifts the midpoint toward the side with less displayed liquidity. Positive trade imbalance means buyer-initiated volume dominated the interval.

In [4]:
features = pd.DataFrame(index=data.index)

# Timing fields make it possible to preserve when the market event occurred
# and when this computer actually observed the book used for the row.
features["event_time"] = data["event_time"]
features["source_received_at"] = data["source_received_at"]
features["event_to_receive_ms"] = data["event_to_receive_ms"]

# Direct book state.
for column in [
    "best_bid", "best_ask", "best_bid_size", "best_ask_size",
    "mid_price", "spread", "bid_depth_10", "ask_depth_10", "imbalance_10",
]:
    features[column] = data[column]

features["spread_bps"] = features["spread"] / features["mid_price"] * 10_000
features["total_depth_10"] = features["bid_depth_10"] + features["ask_depth_10"]
bbo_size = features["best_bid_size"] + features["best_ask_size"]
features["microprice"] = (
    features["best_ask"] * features["best_bid_size"]
    + features["best_bid"] * features["best_ask_size"]
) / bbo_size.replace(0, np.nan)
features["microprice_edge_bps"] = (
    features["microprice"] - features["mid_price"]
) / features["mid_price"] * 10_000

# Backward-looking price and book changes.
log_mid = np.log(features["mid_price"])
features["past_mid_return_1s"] = log_mid.diff(1)
features["past_mid_return_5s"] = log_mid.diff(5)
features["past_mid_return_30s"] = log_mid.diff(30)
features["imbalance_change_1s"] = features["imbalance_10"].diff(1)
features["imbalance_mean_5s"] = features["imbalance_10"].rolling(5, min_periods=1).mean()
features["depth_log_change_1s"] = np.log(features["total_depth_10"]).diff(1)
features["volatility_30s"] = features["past_mid_return_1s"].rolling(30, min_periods=10).std()

# Current and rolling trade-flow features.
features["trade_count"] = data["trade_count"]
features["trade_volume"] = data["trade_volume"]
features["signed_trade_volume"] = data["signed_trade_volume"]
features["trade_imbalance_1s"] = (
    data["signed_trade_volume"] / data["trade_volume"].replace(0, np.nan)
).fillna(0)
rolling_signed = data["signed_trade_volume"].rolling(5, min_periods=1).sum()
rolling_volume = data["trade_volume"].rolling(5, min_periods=1).sum()
features["trade_imbalance_5s"] = (rolling_signed / rolling_volume.replace(0, np.nan)).fillna(0)
features["trade_vwap_edge_bps"] = (
    data["trade_vwap"] - features["mid_price"]
) / features["mid_price"] * 10_000

features = features.replace([np.inf, -np.inf], np.nan)
display(features.head())

,event_time,source_received_at,event_to_receive_ms,best_bid,best_ask,best_bid_size,best_ask_size,mid_price,spread,bid_depth_10,...,imbalance_change_1s,imbalance_mean_5s,depth_log_change_1s,volatility_30s,trade_count,trade_volume,signed_trade_volume,trade_imbalance_1s,trade_imbalance_5s,trade_vwap_edge_bps
decision_time,,,,,,,,,,,,,,,,,,,,,
2026-10-09 10:38:47+00:00,2026-10-09 10:38:46.321411+00:00,2026-10-09 10:38:46.505635+00:00,184.224,109.6,109.62,63.133094,80.545522,109.61,0.02,2537.544435,...,NaN,0.145982,NaN,NaN,0,0.000000,0.000000,0.0,0.0,NaN
2026-10-09 10:38:48+00:00,2026-10-09 10:38:47.536561+00:00,2026-10-09 10:38:47.554103+00:00,17.542,109.6,109.62,63.133094,41.610000,109.61,0.02,2534.097085,...,0.045718,0.168841,-0.040478,NaN,0,0.000000,0.000000,0.0,0.0,NaN
2026-10-09 10:38:49+00:00,2026-10-09 10:38:48.477619+00:00,2026-10-09 10:38:48.583365+00:00,105.746,109.6,109.62,69.280735,41.610000,109.61,0.02,2753.824667,...,0.170884,0.233422,-0.050849,NaN,0,0.000000,0.000000,0.0,0.0,NaN
2026-10-09 10:38:50+00:00,2026-10-09 10:38:49.531141+00:00,2026-10-09 10:38:49.544308+00:00,13.167,109.6,109.62,63.972227,42.658605,109.61,0.02,2522.030757,...,-0.099067,0.240946,-0.012443,NaN,1,2.659022,2.659022,1.0,1.0,0.912326
2026-10-09 10:38:51+00:00,2026-10-09 10:38:50.496621+00:00,2026-10-09 10:38:50.545895+00:00,49.274,109.6,109.62,63.972227,42.549136,109.61,0.02,2856.709737,...,0.006313,0.246723,0.119622,NaN,1,0.109469,0.109469,1.0,1.0,0.912326


## Create future-return targets

Targets deliberately use future mid-prices and must never be included as model inputs. For example, `target_mid_return_5s` is the log return from the current decision time to five seconds later.

In [5]:
target_columns = []
for horizon in TARGET_HORIZONS:
    column = f"target_mid_return_{horizon}s"
    features[column] = log_mid.shift(-horizon) - log_mid
    target_columns.append(column)

feature_columns = [
    "spread_bps", "imbalance_10", "imbalance_change_1s", "imbalance_mean_5s",
    "total_depth_10", "depth_log_change_1s", "microprice_edge_bps",
    "past_mid_return_1s", "past_mid_return_5s", "past_mid_return_30s",
    "volatility_30s", "trade_count", "trade_volume",
    "signed_trade_volume", "trade_imbalance_1s", "trade_imbalance_5s",
]
model_data = features.dropna(subset=feature_columns + target_columns).copy()

print(f"Complete modeling rows: {len(model_data):,}")
display(model_data[feature_columns[:6] + target_columns].head())

Complete modeling rows: 10,740


,spread_bps,imbalance_10,imbalance_change_1s,imbalance_mean_5s,total_depth_10,depth_log_change_1s,target_mid_return_1s,target_mid_return_5s,target_mid_return_30s
decision_time,,,,,,,,,
2026-10-09 10:39:17+00:00,1.825151,0.430512,-0.027174,0.374427,5197.060569,0.068337,-0.000183,-0.000274,0.000091
2026-10-09 10:39:18+00:00,1.825484,0.035951,-0.394561,0.300491,3852.979297,-0.299246,0.000000,-0.000091,0.000183
2026-10-09 10:39:19+00:00,1.825484,0.138014,0.102063,0.263044,4528.997289,0.161654,0.000046,-0.000091,0.000183
2026-10-09 10:39:20+00:00,0.912700,0.166669,0.028656,0.245766,3829.986614,-0.167639,-0.000046,-0.000046,0.000228
2026-10-09 10:39:21+00:00,1.825484,0.262525,0.095856,0.206734,4443.360974,0.148550,-0.000091,0.000000,0.000274


## Initial predictiveness checks

Rank correlations and imbalance buckets are exploratory diagnostics, not evidence of a profitable strategy. Signals should eventually be evaluated on later, untouched captures with fees, spread, latency, and realistic fills.

In [6]:
# Rank first and then use ordinary correlation, avoiding an additional SciPy dependency.
ranked = model_data[feature_columns + target_columns].rank()
rank_correlations = ranked.corr().loc[feature_columns, target_columns]
display(rank_correlations.round(4))

imbalance_bucket = pd.qcut(model_data["imbalance_10"], 5, labels=False, duplicates="drop")
imbalance_summary = (
    model_data.assign(imbalance_bucket=imbalance_bucket)
    .groupby("imbalance_bucket", observed=True)
    .agg(
        rows=("imbalance_10", "size"),
        mean_imbalance=("imbalance_10", "mean"),
        mean_future_return_1s=("target_mid_return_1s", "mean"),
        mean_future_return_5s=("target_mid_return_5s", "mean"),
        mean_future_return_30s=("target_mid_return_30s", "mean"),
    )
)
display(imbalance_summary)

,target_mid_return_1s,target_mid_return_5s,target_mid_return_30s
spread_bps,0.0126,0.0354,0.0441
imbalance_10,0.0952,0.0604,0.0005
imbalance_change_1s,0.0936,0.0388,0.0238
imbalance_mean_5s,0.0395,0.0476,-0.0231
total_depth_10,0.0101,0.0103,0.0245
depth_log_change_1s,0.0135,0.0109,-0.0024
microprice_edge_bps,0.1333,0.1143,0.0629
past_mid_return_1s,-0.0108,0.0232,0.0078
past_mid_return_5s,0.0199,0.0715,0.0286
past_mid_return_30s,0.0126,0.0213,0.0243


,rows,mean_imbalance,mean_future_return_1s,mean_future_return_5s,mean_future_return_30s
imbalance_bucket,,,,,
0,2148,-0.092491,-0.000013,-0.000018,2.713628e-07
1,2148,0.040138,-0.000006,-0.000012,-1.739347e-05
2,2148,0.121973,-0.000002,-0.000006,-9.032704e-08
3,2148,0.201638,0.000004,0.000004,1.176216e-05
4,2148,0.321717,0.000016,0.000027,-2.036398e-05


## Chronological split and save

Never randomly shuffle market data. The first 70% is a development set and the final 30% is a preliminary test set. A gap equal to the longest target horizon is removed from the end of the development set so its targets cannot use prices from the test period.

In [7]:
split_index = int(len(model_data) * 0.70)
test = model_data.iloc[split_index:]
test_start = test.index.min()
purge = pd.Timedelta(seconds=max(TARGET_HORIZONS))
train = model_data[model_data.index < test_start - purge]
print(f"Purged the {purge.total_seconds():.0f}s before the test period from development data")
print(f"Development: {len(train):,} rows, {train.index.min()} through {train.index.max()}")
print(f"Test:        {len(test):,} rows, {test.index.min()} through {test.index.max()}")

FEATURES_PATH = DATA_DIR / f"coinbase_{PRODUCT_ID}_features_{capture_stamp}.csv"
features.reset_index().to_csv(FEATURES_PATH, index=False)
print(f"Saved {len(features):,} feature rows to {FEATURES_PATH.resolve()}")

Purged the 30s before the test period from development data
Development: 7,487 rows, 2026-10-09 10:39:17+00:00 through 2026-10-09 12:44:03+00:00
Test:        3,223 rows, 2026-10-09 12:44:34+00:00 through 2026-10-09 13:38:16+00:00
Saved 10,800 feature rows to /Users/hankrugg/Fordham/Fall26/Crypto/trading/data/ticks/coinbase_SOL-USD_features_20261009T133847Z.csv
